# 2-d Adaptive Sampling

We wish to adaptively sample a GP like $f \sim \text{GP}(\mu, k)$ such that $f: \mathbb{R}^n \rightarrow \mathbb{R}$ where $\mu$ and $k$ are the prior mean and covariance kernel, respectively.

Similarly to the case in which $f: \mathbb{R} \rightarrow \mathbb{R}$, we (iteratively) update the posterior with the point in our domain associated with highest variance.

In [ ]:
import matplotlib.axes as mpl_axes
import matplotlib.cm as mpl_cm
import matplotlib.colors as mpl_colors
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.insert(0, '../')

import UncertainSCI.gp as gp


MESH_KWARGS = dict(
    truth = dict(
        vmin=-3.,
        vmax=3.,
        cmap='Spectral_r'
    ),
    mean = dict(
        vmin=-3.,
        vmax=3.,
        cmap='Spectral_r'
    ),
    variance = dict(
        vmin=0.,
        vmax=3.,
        cmap='RdYlGn_r'
    ),
    error = dict(vmin=0.,
        vmax=3.,
        cmap='RdYlGn_r')
)
FIGSIZE = (6.4, 4.8)
PLOT_EVERY = 10

N_SMP = int(1e5)

E1_PTS = 40
E1LIM = (0, 2)
E2_PTS = 20
E2LIM = (0, 1)

CMAP = mpl_cm.get_cmap('cool')
CMAP.set_bad('#000000')
CMAP.set_over('#ff0000')
NORM = mpl_colors.Normalize(vmin=0., vmax=3.)
M = mpl_cm.ScalarMappable(norm=NORM, cmap=CMAP)


Define our domain and some plotting helpers:

In [ ]:
E1 = np.linspace(*E1LIM, E1_PTS)
E2 = np.linspace(*E2LIM, E2_PTS)
E1MG, E2MG = np.meshgrid(E1, E2)

E1_BIG = np.linspace(*E1LIM, 100 * E1_PTS)
E2_BIG = np.linspace(*E2LIM, 100 * E2_PTS)
E1MG_BIG, E2MG_BIG = np.meshgrid(E1_BIG, E2_BIG)

def draw_distribution(axes: np.ndarray[mpl_axes.Axes], g: gp.ScalarGaussianProcess,
                      which='posterior'):
    if not (which == 'posterior' or which == 'prior'):
        raise ValueError("`which` parameter must be 'posterior' (default) or 'prior'")
    
    x = np.stack((E1MG, E2MG), axis=-1).reshape((-1, 2))

    what = ('truth', 'mean', 'variance', 'error')
    truth = f(x).reshape((E2_PTS, E1_PTS))
    mean = (g.mu_posterior(x) if which == 'posterior' else g.mu(x)).reshape((E2_PTS, E1_PTS))
    var = np.diag((g.k_posterior(x) if which == 'posterior' else g.k(x))).reshape((E2_PTS, E1_PTS))
    error = np.abs(truth - mean)

    for w, y, ax in zip(what, (truth, mean, var, error), axes):
        m = ax.pcolormesh(E1MG, E2MG, y, **MESH_KWARGS[w])
        plt.colorbar(m, ax=ax)
        ax.set_aspect('equal')
        ax.set_xlim(*E1LIM)
        ax.set_ylim(*E2LIM)
        ax.set_title(w.title())


## True Function and Noisy Observations

Define the true function and a function that yields noisy observations of that true function.

In particular, for this example, let the true function $f: \mathbb{R}^2 \rightarrow \mathbb{R}$.  Then noisy observations $\hat{f}(x) = f(x) + \epsilon$ of the true function.  As defined below, $\epsilon \sim \text{N}(0, \sigma^2)$ where $\sigma^2 \sim \text{LogNormal}(-1, 0.8)$.  In general, however, GP methods are able to handle noise that is itself a function of the spatial coordinate, or even more exotic scenarios.


In [ ]:
def f(x):
    a = 2
    return np.sin(a * 2 * np.pi * x[..., 0]**2) * x[..., 1]

def f_random(x: np.ndarray | float):
    fx = f(x)
    s = sigma(fx)
    return fx + (s.flatten() * np.random.normal(0, 1, s.size)).reshape(s.shape), s

def sigma(fx: np.ndarray):
    # (mu, sigma) = (-1, 0.8) chosen simply for looking nice
    return np.random.lognormal(-1, 0.8, fx.shape)


We plot the true function below:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FIGSIZE[0] * 2, FIGSIZE[1]))

for j, tt, ff in zip((0, 1), ('Truth', 'Noisy'), (f, lambda x: f_random(x)[0])):
    ax = axes[j]

    m = ax.pcolormesh(E1MG_BIG, E2MG_BIG, ff(np.stack((E1MG_BIG, E2MG_BIG), axis=-1)), **MESH_KWARGS['truth'])
    plt.colorbar(m, ax=ax)
    ax.set_aspect('equal')
    ax.set_xlim(*E1LIM)
    ax.set_ylim(*E2LIM)
    ax.set_title(tt)

plt.tight_layout()
plt.show()


## Defining GP and Prior Mean and Covariance
Define the prior Gaussian process and create the ScalarGaussianProcess object:

In [ ]:
mu = gp.wrapper.ScalarFunction(dim=2, f=lambda x: np.zeros(len(x)))  # zero mean
k = gp.kernel.SquareExponential(dim=2, gamma=0.2)  # square exponential kernel with scale 1
g = gp.ScalarGaussianProcess(mu, k)  # GP defined from these mu, k


We can plot a realizations of the prior:

In [ ]:
X = np.stack((E1MG, E2MG), axis=-1).reshape((-1, 2))
y = g.sample_prior(X).reshape((E2_PTS, E1_PTS))


fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)

m = ax.pcolormesh(E1MG, E2MG, y, **MESH_KWARGS['mean'])
plt.colorbar(m, ax=ax)
ax.set_aspect('equal')
ax.set_xlim(*E1LIM)
ax.set_ylim(*E2LIM)

plt.tight_layout()
plt.show()


Plot statistics of many realizations of the prior:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(FIGSIZE[0] * 4, FIGSIZE[1]))
draw_distribution(axes, g, which='prior')
plt.tight_layout()
plt.show()


## Conditioned GP and Posterior Mean and Covariance
Now we condition the GP on realizations of the random function $\hat{f}$ (i.e., `f_random` above):

In [ ]:
N_STRT = 10

x_obs = np.stack((E1LIM[0] + np.ptp(E1LIM) * np.random.rand(N_STRT),
                  E2LIM[0] + np.ptp(E2LIM) * np.random.rand(N_STRT)), axis=-1)
y_obs, s_obs = f_random(x_obs)
g.condition(x_obs, y_obs, s_obs)


Before conducting any iterative procedure, we ought to plot our initial "guess" at the true function given the observation pairs $(x_{obs}, y_{obs})$ associated with variances $\sigma_{obs}$, which we do below.

The cyan-pink colorbar indicates sample variance, and the blue-green-yellow-red colorbar indicates function height:


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(FIGSIZE[0] * 4, FIGSIZE[1]))
draw_distribution(axes, g)
for ax in axes[[0, 2]]:
    ax.scatter(*(x_obs.T), marker='D', c=M.to_rgba(s_obs), s=10)
    plt.colorbar(M, ax=ax, extend='max')
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()


We now iteratively update our posterior with observations at the coordinate associated with the highest variance.

In the figures below, the green vertical bar indicates the *coordinate* at which to choose the next sample.  In the figure that follows, the sample chosen at this location is shown with its associated (sampled) variance.

In [ ]:
N_RUN = 10
PLOT_EVERY = 1

for i in range(N_RUN):
    x = X[np.argsort(np.diag(g.k_posterior(X)))[-1:]]
    y, s = f_random(x)

    if (i + 1) % PLOT_EVERY == 0:
        print(f'Sample {i + 1}:')

        fig, axes = plt.subplots(1, 4, figsize=(FIGSIZE[0] * 4, FIGSIZE[1]))
        draw_distribution(axes, g)
        for ax in axes[[0, 2]]:
            ax.scatter(*(x_obs.T), marker='D', c=M.to_rgba(s_obs), s=10)
            ax.scatter(*(x.T), marker='X', color='#00ff00', s=300)
            plt.colorbar(M, ax=ax, extend='max')
            ax.set_aspect('equal')
        plt.tight_layout()
        plt.show()

    x_obs = np.concatenate((x_obs, x), axis=0)
    y_obs = np.concatenate((y_obs, y), axis=0)    
    s_obs = np.concatenate((s_obs, s), axis=0)
    g.condition(x_obs, y_obs, s_obs)

    if (i + 1) % PLOT_EVERY == 0:
        fig, axes = plt.subplots(1, 4, figsize=(FIGSIZE[0] * 4, FIGSIZE[1]))
        draw_distribution(axes, g)
        for ax in axes[[0, 2]]:
            ax.scatter(*(x_obs[:-1].T), marker='D', c=M.to_rgba(s_obs[:-1]), s=10)
            ax.scatter(*(x_obs[-1].T), marker='o', color=M.to_rgba(s_obs[-1]), s=200)
            plt.colorbar(M, ax=ax, extend='max')
            ax.set_aspect('equal')
        plt.tight_layout()
        plt.show()

        print('\n' * 3, end='')
